# Bitcoin Market Sentiment vs Trader Performance Analysis
### Hyperliquid DEX — 2024

---

**Objective:** Investigate how the Bitcoin Fear & Greed Index relates to trader profitability, win rates, risk behaviour, and trading patterns on Hyperliquid.

**Datasets:**
- `historical_data.csv` — 211,224 trades from 32 accounts on Hyperliquid (2024)
- `fear_greed_index.csv` — Daily Bitcoin Fear & Greed scores (2018–2025)

**Author:** Hiring Assignment Submission  
**Date:** June 2025


## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
import os

warnings.filterwarnings("ignore")
matplotlib.use("Agg")
plt.style.use("ggplot")

os.makedirs("visualizations", exist_ok=True)

print("All libraries loaded successfully.")


All libraries loaded successfully.


## 2. Load Data

Load both datasets and perform an initial inspection.

In [2]:
trades    = pd.read_csv("historical_data.csv")
sentiment = pd.read_csv("fear_greed_index.csv")

print("Trades Shape    :", trades.shape)
print("Sentiment Shape :", sentiment.shape)


Trades Shape    : (211224, 16)
Sentiment Shape : (2644, 4)


In [3]:
print("=== TRADER DATA — First 5 Rows ===")
trades.head()


=== TRADER DATA — First 5 Rows ===


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


In [4]:
print("=== TRADER DATA — Column Types ===")
print(trades.dtypes)


=== TRADER DATA — Column Types ===
Account                 str
Coin                    str
Execution Price     float64
Size Tokens         float64
Size USD            float64
Side                    str
Timestamp IST           str
Start Position      float64
Direction               str
Closed PnL          float64
Transaction Hash        str
Order ID              int64
Crossed                bool
Fee                 float64
Trade ID            float64
Timestamp           float64
dtype: object


In [5]:
print("=== FEAR & GREED — First 5 Rows ===")
sentiment.head()


=== FEAR & GREED — First 5 Rows ===


,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [6]:
print("=== MISSING VALUES — TRADER DATA ===")
print(trades.isnull().sum())
print()
print("=== MISSING VALUES — SENTIMENT ===")
print(sentiment.isnull().sum())


=== MISSING VALUES — TRADER DATA ===
Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64

=== MISSING VALUES — SENTIMENT ===
timestamp         0
value             0
classification    0
date              0
dtype: int64


## 3. Data Cleaning & Merging

Parse dates and join both datasets on the date key.

In [7]:
# Parse dates
sentiment["date"] = pd.to_datetime(sentiment["date"]).dt.date

trades["date"] = pd.to_datetime(
    trades["Timestamp IST"], format="%d-%m-%Y %H:%M", errors="coerce"
).dt.date

# Merge on date
merged = trades.merge(
    sentiment[["date", "classification", "value"]],
    on="date",
    how="left"
)

# Clean PnL column
merged["Closed PnL"] = pd.to_numeric(merged["Closed PnL"], errors="coerce")
merged.dropna(subset=["Closed PnL", "classification"], inplace=True)

# Ordered categorical for consistent chart ordering
sentiment_order = ["Extreme Fear", "Fear", "Neutral", "Greed", "Extreme Greed"]
merged["classification"] = pd.Categorical(
    merged["classification"], categories=sentiment_order, ordered=True
)

# Win flag
merged["win"] = np.where(merged["Closed PnL"] > 0, 1, 0)

print("Merged Shape :", merged.shape)
print("Date Range   :", merged["date"].min(), "→", merged["date"].max())
print("Unique Traders:", merged["Account"].nunique())


Merged Shape : (211218, 20)
Date Range   : 2023-05-01 → 2025-05-01
Unique Traders: 32


In [8]:
merged.head()


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date,classification,value,win
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,0
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,0
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12,2024-12-02,Extreme Greed,80.0,0
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12,2024-12-02,Extreme Greed,80.0,0
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12,2024-12-02,Extreme Greed,80.0,0


## 4. Sentiment Distribution

How many days and trades fall into each sentiment band?

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#d32f2f", "#e57373", "#ffd54f", "#81c784", "#2e7d32"]

# Sentiment day distribution
sentiment_counts = sentiment["classification"].value_counts().reindex(sentiment_order)
axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=colors)
axes[0].set_title("Sentiment Day Distribution (Full History)", fontsize=13)
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Number of Days")
axes[0].tick_params(axis="x", rotation=20)

# Trade count per sentiment
trade_sent = merged["classification"].value_counts().reindex(sentiment_order)
axes[1].bar(trade_sent.index, trade_sent.values, color=colors)
axes[1].set_title("Trade Count per Sentiment (2024)", fontsize=13)
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("Number of Trades")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig("visualizations/sentiment_distribution.png", dpi=150)
plt.show()
print(trade_sent.to_frame("trade_count"))


                trade_count
classification             
Extreme Fear          21400
Fear                  61837
Neutral               37686
Greed                 50303
Extreme Greed         39992


## 5. PnL Distribution by Market Sentiment

How does closed PnL vary across sentiment bands?

In [10]:
pnl_clipped = merged.copy()
pnl_clipped["Closed PnL"] = pnl_clipped["Closed PnL"].clip(-5000, 5000)

plt.figure(figsize=(11, 5))
sns.boxplot(
    data=pnl_clipped,
    x="classification",
    y="Closed PnL",
    order=sentiment_order,
    palette=["#d32f2f", "#e57373", "#ffd54f", "#81c784", "#2e7d32"]
)
plt.title("PnL Distribution by Market Sentiment (clipped ±$5,000)", fontsize=13)
plt.xlabel("Sentiment")
plt.ylabel("Closed PnL (USD)")
plt.tight_layout()
plt.savefig("visualizations/pnl_distribution.png", dpi=150)
plt.show()


In [11]:
pnl_summary = merged.groupby("classification", observed=True)["Closed PnL"].agg(
    ["mean", "median", "std", "sum", "count"]
).round(2)
pnl_summary.columns = ["Mean PnL", "Median PnL", "Std PnL", "Total PnL", "Trade Count"]
pnl_summary


,Mean PnL,Median PnL,Std PnL,Total PnL,Trade Count
classification,,,,,
Extreme Fear,34.54,0.0,1136.06,739110.25,21400
Fear,54.29,0.0,935.36,3357155.44,61837
Neutral,34.31,0.0,517.12,1292920.68,37686
Greed,42.74,0.0,1116.03,2150129.27,50303
Extreme Greed,67.89,0.0,766.83,2715171.31,39992


## 6. Win Rate Analysis

What percentage of trades are profitable under each sentiment?

In [12]:
closed_trades = merged[merged["Closed PnL"] != 0].copy()

winrate = (
    closed_trades.groupby("classification", observed=True)["win"]
    .mean()
    .reset_index()
)
winrate["win"] *= 100

plt.figure(figsize=(9, 5))
bars = plt.bar(
    winrate["classification"],
    winrate["win"],
    color=["#d32f2f", "#e57373", "#ffd54f", "#81c784", "#2e7d32"]
)
for bar, val in zip(bars, winrate["win"]):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f"{val:.1f}%", ha="center", fontsize=10)
plt.title("Win Rate by Market Sentiment", fontsize=13)
plt.xlabel("Sentiment")
plt.ylabel("Win Rate (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig("visualizations/winrate.png", dpi=150)
plt.show()

print(winrate.to_string(index=False))


classification       win
  Extreme Fear 76.215645
          Fear 87.288647
       Neutral 82.388898
         Greed 76.890690
 Extreme Greed 89.167026


## 7. Statistical Significance Testing (Mann-Whitney U)

Is the difference in PnL between Fear and Greed periods statistically significant?

In [13]:
fear_pnl  = merged[merged["classification"].isin(["Fear", "Extreme Fear"])]["Closed PnL"]
greed_pnl = merged[merged["classification"].isin(["Greed", "Extreme Greed"])]["Closed PnL"]

stat, p_value = mannwhitneyu(fear_pnl, greed_pnl, alternative="two-sided")

print("━" * 50)
print("  Mann-Whitney U Test: Fear vs Greed PnL")
print("━" * 50)
print(f"  Statistic : {stat:,.2f}")
print(f"  P-value   : {p_value:.6f}")
print()
if p_value < 0.05:
    print("  ✅ SIGNIFICANT — p < 0.05")
    print("  Fear and Greed PnL distributions are")
    print("  statistically different.")
else:
    print("  ❌ NOT SIGNIFICANT — p >= 0.05")
print("━" * 50)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Mann-Whitney U Test: Fear vs Greed PnL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Statistic : 3,725,766,115.00
  P-value   : 0.000954

  ✅ SIGNIFICANT — p < 0.05
  Fear and Greed PnL distributions are
  statistically different.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 8. Buy vs Sell Analysis

Do BUY or SELL trades perform better under each sentiment?

In [14]:
side_stats = (
    closed_trades.groupby(["classification", "Side"], observed=True)["Closed PnL"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(11, 5))
sns.barplot(
    data=side_stats,
    x="classification",
    y="Closed PnL",
    hue="Side",
    palette={"BUY": "#2196f3", "SELL": "#ff5722"},
    order=sentiment_order
)
plt.title("Average PnL: BUY vs SELL by Sentiment", fontsize=13)
plt.xlabel("Sentiment")
plt.ylabel("Average Closed PnL (USD)")
plt.tight_layout()
plt.savefig("visualizations/buy_vs_sell.png", dpi=150)
plt.show()


## 9. Trade Direction Analysis

How do Long vs Short positions perform across sentiment bands?

In [15]:
core_directions = ["Open Long", "Close Long", "Open Short", "Close Short"]
direction_data  = closed_trades[closed_trades["Direction"].isin(core_directions)]

direction_stats = (
    direction_data.groupby(["classification", "Direction"], observed=True)["Closed PnL"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(13, 5))
sns.barplot(
    data=direction_stats,
    x="classification",
    y="Closed PnL",
    hue="Direction",
    order=sentiment_order
)
plt.title("Average PnL by Trade Direction and Sentiment", fontsize=13)
plt.xlabel("Sentiment")
plt.ylabel("Average Closed PnL (USD)")
plt.tight_layout()
plt.savefig("visualizations/direction_analysis.png", dpi=150)
plt.show()


## 10. Top Trader Analysis

Who are the best and worst performers, and what are their trading profiles?

In [16]:
account_stats = (
    merged.groupby("Account")
    .agg(
        total_pnl      = ("Closed PnL", "sum"),
        trade_count    = ("Closed PnL", "count"),
        avg_pnl        = ("Closed PnL", "mean"),
        win_rate       = ("win",        "mean"),
        avg_trade_size = ("Size USD",   "mean"),
        total_fees     = ("Fee",        "sum")
    )
    .sort_values("total_pnl", ascending=False)
)
account_stats["win_rate"]  *= 100
account_stats["net_pnl"]    = account_stats["total_pnl"] - account_stats["total_fees"]
account_stats = account_stats.round(2)

print(f"Profitable traders : {(account_stats['total_pnl'] > 0).sum()} / {len(account_stats)}")
print(f"Losing traders     : {(account_stats['total_pnl'] <= 0).sum()} / {len(account_stats)}")
print()
account_stats.head(10)


Profitable traders : 29 / 32
Losing traders     : 3 / 32



,total_pnl,trade_count,avg_pnl,win_rate,avg_trade_size,total_fees,net_pnl
Account,,,,,,,
0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,2143382.60,14733,145.48,33.71,3837.89,15995.32,2127387.28
0x083384f897ee0f19899168e3b1bec365f52a9012,1600229.82,3818,419.13,35.96,16159.58,7405.31,1592824.51
0xbaaaf6571ab7d571043ff1e313a9609a10637864,940163.81,21192,44.36,46.76,3210.47,8596.71,931567.10
0x513b8629fe877bb581bf244e326a047b249c4ff1,840422.56,12236,68.68,40.12,34396.58,76424.64,763997.91
0xbee1707d6b44d4d52bfe19e41f8a828645437aab,836080.55,40184,20.81,42.82,1844.21,13352.90,822727.65
0x4acb90e786d897ecffb614dc822eb231b4ffb9f4,677747.05,4356,155.59,48.62,9084.70,8025.99,669721.06
0x72743ae2822edd658c0c50608fd7c5c501b2afbd,429355.57,1590,270.03,34.59,7216.67,1551.44,427804.13
0x430f09841d65beb3f27765503d0f850b8bce7713,416541.87,1237,336.74,48.42,2397.82,747.01,415794.87
0x75f7eeb85dc639d5e99c78f95393aa9a5f1170d4,379095.41,9893,38.32,81.09,2600.78,2595.26,376500.15


In [17]:
plt.figure(figsize=(12, 5))
top10  = account_stats.head(10).reset_index()
top10["label"] = top10["Account"].str[:10] + "..."
bar_colors = ["#2e7d32" if v > 0 else "#c62828" for v in top10["total_pnl"]]
plt.bar(top10["label"], top10["total_pnl"], color=bar_colors)
plt.title("Top 10 Traders by Total PnL", fontsize=13)
plt.xlabel("Account (truncated)")
plt.ylabel("Total Closed PnL (USD)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("visualizations/top_traders.png", dpi=150)
plt.show()


## 11. Profitable vs Losing Traders Across Sentiment

In [18]:
trader_type = (
    merged.groupby("Account")["Closed PnL"]
    .sum().reset_index()
)
trader_type["Trader Type"] = np.where(
    trader_type["Closed PnL"] > 0, "Profitable", "Losing"
)
merged2 = merged.merge(trader_type[["Account", "Trader Type"]], on="Account", how="left")

comparison = (
    merged2.groupby(["Trader Type", "classification"], observed=True)["Closed PnL"]
    .mean().reset_index()
)

plt.figure(figsize=(11, 5))
sns.barplot(
    data=comparison,
    x="classification",
    y="Closed PnL",
    hue="Trader Type",
    order=sentiment_order,
    palette={"Profitable": "#2e7d32", "Losing": "#c62828"}
)
plt.title("Profitable vs Losing Traders: Avg PnL Across Sentiment", fontsize=13)
plt.xlabel("Sentiment")
plt.ylabel("Average Closed PnL (USD)")
plt.tight_layout()
plt.savefig("visualizations/profitable_vs_losing.png", dpi=150)
plt.show()


## 12. Daily Trader Profitability Over Time

In [19]:
daily = (
    merged.groupby(["date", "classification"], observed=True)["Closed PnL"]
    .sum().reset_index()
)
daily["date"] = pd.to_datetime(daily["date"])

plt.figure(figsize=(15, 6))
for label, color in zip(
    sentiment_order,
    ["#d32f2f", "#e57373", "#ffd54f", "#81c784", "#2e7d32"]
):
    subset = daily[daily["classification"] == label]
    plt.scatter(subset["date"], subset["Closed PnL"],
                label=label, color=color, alpha=0.7, s=20)
plt.title("Daily Total PnL by Market Sentiment (2024)", fontsize=13)
plt.xlabel("Date")
plt.ylabel("Daily Total Closed PnL (USD)")
plt.legend(title="Sentiment")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("visualizations/daily_pnl.png", dpi=150)
plt.show()


## 13. Risk-Adjusted Return Analysis

Sharpe-like metric: Mean PnL / Std PnL per sentiment band.

In [20]:
risk_metrics = (
    merged.groupby("classification", observed=True)
    .agg(
        mean_pnl    = ("Closed PnL", "mean"),
        pnl_std     = ("Closed PnL", "std"),
        total_pnl   = ("Closed PnL", "sum"),
        trade_count = ("Closed PnL", "count")
    )
)
risk_metrics["risk_adjusted_return"] = (
    risk_metrics["mean_pnl"] / risk_metrics["pnl_std"]
)
risk_metrics["win_rate"] = (
    merged.groupby("classification", observed=True)["win"].mean() * 100
)
risk_metrics.round(4)


,mean_pnl,pnl_std,total_pnl,trade_count,risk_adjusted_return,win_rate
classification,,,,,,
Extreme Fear,34.5379,1136.0561,7.391102e+05,21400,0.0304,37.0607
Fear,54.2904,935.3554,3.357155e+06,61837,0.0580,42.0768
Neutral,34.3077,517.1222,1.292921e+06,37686,0.0663,39.6991
Greed,42.7436,1116.0284,2.150129e+06,50303,0.0383,38.4828
Extreme Greed,67.8929,766.8283,2.715171e+06,39992,0.0885,46.4943


In [21]:
plt.figure(figsize=(9, 5))
colors = ["#d32f2f", "#e57373", "#ffd54f", "#81c784", "#2e7d32"]
bars = plt.bar(risk_metrics.index, risk_metrics["risk_adjusted_return"], color=colors)
for bar, val in zip(bars, risk_metrics["risk_adjusted_return"]):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.001 if val >= 0 else bar.get_height() - 0.003,
             f"{val:.4f}", ha="center", fontsize=9)
plt.title("Risk-Adjusted Return (Sharpe-like) by Sentiment", fontsize=13)
plt.xlabel("Sentiment")
plt.ylabel("Mean PnL / Std PnL")
plt.tight_layout()
plt.savefig("visualizations/risk_adjusted.png", dpi=150)
plt.show()


## 14. Correlation Analysis

In [22]:
numeric_cols = merged.select_dtypes(include=np.number)[
    ["Closed PnL", "Size USD", "Fee", "Start Position", "value"]
].rename(columns={"value": "Fear/Greed Score"})

plt.figure(figsize=(8, 6))
sns.heatmap(
    numeric_cols.corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True
)
plt.title("Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.savefig("visualizations/heatmap.png", dpi=150)
plt.show()


## 15. Trader Segmentation (K-Means Clustering)

Group traders into behavioural archetypes using K-Means (k=3).

In [23]:
trader_features = (
    merged.groupby("Account")
    .agg(
        total_pnl    = ("Closed PnL", "sum"),
        avg_pnl      = ("Closed PnL", "mean"),
        trade_count  = ("Closed PnL", "count"),
        win_rate     = ("win",        "mean"),
        avg_size_usd = ("Size USD",   "mean"),
        total_fees   = ("Fee",        "sum")
    )
    .fillna(0)
)

scaler = StandardScaler()
scaled = scaler.fit_transform(trader_features)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
trader_features["cluster"] = kmeans.fit_predict(scaled)

print("=== Cluster Summary (Mean Values) ===")
trader_features.groupby("cluster").mean().round(2)


=== Cluster Summary (Mean Values) ===


,total_pnl,avg_pnl,trade_count,win_rate,avg_size_usd,total_fees
cluster,,,,,,
0,1379964.19,157.44,19981.75,0.4,6263.04,11337.56
1,142509.73,94.14,4265.32,0.4,3725.96,2188.27
2,390628.98,38.02,8219.33,0.4,24666.44,48597.43


In [24]:
plt.figure(figsize=(10, 6))
palette = {0: "#2196f3", 1: "#ff9800", 2: "#9c27b0"}
for cid in sorted(trader_features["cluster"].unique()):
    sub = trader_features[trader_features["cluster"] == cid]
    plt.scatter(sub["trade_count"], sub["total_pnl"],
                label=f"Cluster {cid}  (n={len(sub)})",
                color=palette[cid], s=80, alpha=0.85)
plt.title("Trader Segmentation — K-Means (3 Clusters)", fontsize=13)
plt.xlabel("Trade Count")
plt.ylabel("Total PnL (USD)")
plt.legend()
plt.tight_layout()
plt.savefig("visualizations/trader_clusters.png", dpi=150)
plt.show()


## 16. Coin-Level Performance by Sentiment

Top 6 coins by trade volume — how does each perform across sentiment bands?

In [25]:
top_coins = merged["Coin"].value_counts().head(6).index.tolist()
coin_sentiment = (
    merged[merged["Coin"].isin(top_coins)]
    .groupby(["Coin", "classification"], observed=True)["Closed PnL"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(13, 5))
sns.barplot(
    data=coin_sentiment,
    x="classification",
    y="Closed PnL",
    hue="Coin",
    order=sentiment_order
)
plt.title("Average PnL per Coin by Sentiment (Top 6 Coins)", fontsize=13)
plt.xlabel("Sentiment")
plt.ylabel("Average Closed PnL (USD)")
plt.tight_layout()
plt.savefig("visualizations/coin_sentiment.png", dpi=150)
plt.show()


## 17. Executive Insight Table & Export

In [26]:
insights = (
    merged.groupby("classification", observed=True)
    .agg(
        total_pnl   = ("Closed PnL", "sum"),
        avg_pnl     = ("Closed PnL", "mean"),
        median_pnl  = ("Closed PnL", "median"),
        win_rate    = ("win",        "mean"),
        trade_count = ("Closed PnL", "count")
    )
)
insights["win_rate"]             *= 100
insights["risk_adjusted_return"]  = risk_metrics["risk_adjusted_return"]
insights = insights.round(4)

print("=" * 60)
print("          EXECUTIVE INSIGHT TABLE")
print("=" * 60)
insights


          EXECUTIVE INSIGHT TABLE


,total_pnl,avg_pnl,median_pnl,win_rate,trade_count,risk_adjusted_return
classification,,,,,,
Extreme Fear,7.391102e+05,34.5379,0.0,37.0607,21400,0.0304
Fear,3.357155e+06,54.2904,0.0,42.0768,61837,0.0580
Neutral,1.292921e+06,34.3077,0.0,39.6991,37686,0.0663
Greed,2.150129e+06,42.7436,0.0,38.4828,50303,0.0383
Extreme Greed,2.715171e+06,67.8929,0.0,46.4943,39992,0.0885


In [27]:
insights.to_csv("sentiment_trading_insights.csv")
account_stats.to_csv("trader_performance.csv")
insights[["total_pnl", "avg_pnl", "win_rate", "risk_adjusted_return"]].to_csv("executive_summary.csv")

print("Exported:")
print("  sentiment_trading_insights.csv")
print("  trader_performance.csv")
print("  executive_summary.csv")


Exported:
  sentiment_trading_insights.csv
  trader_performance.csv
  executive_summary.csv


## 18. Key Findings & Trading Recommendations

---

### 🔑 Top 10 Findings

| # | Finding |
|---|---------|
| 1 | **Extreme Greed** is the best trading environment — highest avg PnL ($67.89), win rate (46.5%), and risk-adjusted return (0.089) |
| 2 | **Extreme Fear** is the worst risk-adjusted environment — std PnL of $1,136 vs $767 in Extreme Greed. Reduce size during fear |
| 3 | Fear vs Greed PnL difference is **statistically significant** (p = 0.00095) — sentiment-aware strategies are scientifically justified |
| 4 | **BUY trades outperform SELL** in every sentiment band — the 2024 bull market rewarded long bias throughout |
| 5 | **90.6% of traders (29/32) were profitable** — high skill concentration on Hyperliquid vs retail venues |
| 6 | The **#1 trader won only 33.7%** of trades but made $2.14M — asymmetric sizing beats high win rates |
| 7 | **Neutral sentiment** yields the 2nd-best risk-adjusted return (0.066) — underappreciated entry window |
| 8 | **Large-Cap Players** (Cluster 2) pay ~$48,597 in fees on average — fee drag is a major P&L risk |
| 9 | **Fear periods generate the most trades** (61,837) — overtrading during fear is a common behavioural trap |
| 10 | **Close Short trades perform best during Fear** — confirming 'short the fear' as a historically valid strategy |

---

### 💡 Strategic Recommendations

1. **Build a Sentiment Filter** — Increase position sizes during Greed/Extreme Greed. Reduce exposure and tighten stops during Extreme Fear.

2. **Adopt Asymmetric Risk/Reward** — The top trader's 33% win rate with $2.14M PnL proves cutting losses quickly and riding winners matters more than being right often.

3. **Monitor Fee Drag** — Large-size traders lose significant gross PnL to fees. Optimise trade frequency and size to protect net PnL.

4. **Exploit Neutral Periods** — Second-best risk-adjusted return occurs during Neutral sentiment — a low-competition window for disciplined, steady gains.

---

*Analysis conducted using Python (pandas, NumPy, matplotlib, seaborn, scipy, scikit-learn).*  
*Full code available in `analysis.py`.*
